# Q2 — Faster R-CNN Vehicle Detection (scaffold)

This notebook will contain a self-contained implementation for Q2. All helpers will be inline. Steps:

- Dataset preparation and mapping vehicle classes to a single label
- Dataset and DataLoader for detection (targets with boxes and labels)
- Build Faster R-CNN with MobileNetV2 backbone (torchvision) and custom head
- Training loop, evaluation (mAP@50), and visualizations


In [ ]:
# Imports
import os
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Detection utilities will be implemented inline below

In [ ]:
# Dataset class for LSVH-like dataset
import json
from PIL import Image


class LSVHDataset(Dataset):
    def __init__(self, root_dir, annotation_json, transforms=None):
        self.root_dir = Path(root_dir)
        with open(annotation_json, "r") as f:
            self.annotations = json.load(f)
        # annotations expected as dict: { 'image_filename': [ { 'label': str, 'bbox':[x,y,w,h] }, ... ] }
        self.keys = list(self.annotations.keys())
        self.transforms = transforms

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        img_path = self.root_dir / key
        img = Image.open(img_path).convert("RGB")
        objs = self.annotations[key]
        boxes = []
        labels = []
        for o in objs:
            if o.get("label", "").lower() == "dont_care":
                continue
            bbox = o["bbox"]
            x1, y1, w, h = bbox
            boxes.append([x1, y1, x1 + w, y1 + h])
            labels.append(1)  # vehicle
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}
        if self.transforms:
            img = self.transforms(img)
        return img, target

In [ ]:
from pathlib import Path
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator


# Build Faster R-CNN with MobileNetV2 backbone
def get_fasterrcnn_mobilenet(num_classes=2, pretrained_backbone=True):
    # Load mobilenet_v2 backbone features
    mobilenet = torchvision.models.mobilenet_v2(pretrained=pretrained_backbone)
    backbone = mobilenet.features
    backbone.out_channels = 1280  # mobilenet_v2 last channel count

    # Anchor generator (sizes and aspect ratios)
    anchor_generator = AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),), aspect_ratios=((0.5, 1.0, 2.0),)
    )

    # RoI pooler
    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=["0"], output_size=7, sampling_ratio=2
    )

    model = FasterRCNN(
        backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
    )
    return model


# Simple training utilities


def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    running_loss = 0.0
    for images, targets in data_loader:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        running_loss += losses.item()
    return running_loss / len(data_loader)


@torch.no_grad()
def evaluate_map_simplified(
    model, data_loader, device, iou_thresh=0.5, score_thresh=0.5
):
    model.eval()
    total = 0
    correct = 0
    for images, targets in data_loader:
        images = list(img.to(device) for img in images)
        outputs = model(images)
        for out, tgt in zip(outputs, targets):
            pred_boxes = out["boxes"].cpu()
            pred_scores = out["scores"].cpu()
            keep = pred_scores > score_thresh
            pred_boxes = pred_boxes[keep]
            gt_boxes = tgt["boxes"]
            if pred_boxes.shape[0] == 0 or gt_boxes.shape[0] == 0:
                continue
            ious = torchvision.ops.box_iou(pred_boxes, gt_boxes)
            if ious.numel() > 0 and torch.max(ious) > iou_thresh:
                correct += 1
            total += 1
    return correct / (total + 1e-8)


# Visualization helper


def visualize_predictions(model, img_pil, device, threshold=0.5):
    model.eval()
    transform = T.ToTensor()
    img_t = transform(img_pil).to(device)
    with torch.no_grad():
        preds = model([img_t])[0]
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(img_pil)
    for box, score in zip(preds["boxes"], preds["scores"]):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box.cpu().numpy().astype(int)
        rect = plt.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, fill=False, color="r", linewidth=2
        )
        ax.add_patch(rect)
        ax.text(
            x1,
            y1 - 5,
            f"{score:.2f}",
            color="white",
            fontsize=12,
            backgroundcolor="red",
        )
    plt.axis("off")
    plt.show()


# Example instantiation (uncomment to run locally):
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = get_fasterrcnn_mobilenet(num_classes=2)
# model.to(device)